# Les ventes DVF 2021-2025 dans DuckDB

Objectif : charger les 5 années DVF de La Réunion dans une base **DuckDB**, réécrire en SQL le filtre `ventes_simples` du notebook 1, vérifier qu'on retrouve les mêmes chiffres, puis suivre l'évolution du prix médian au m².

Les fichiers sont téléchargés au préalable par `python scripts/download_dvf.py`.

## Charger les fichiers dans une table

`read_csv_auto` lit directement les `.csv.gz`. Le motif `974_*.csv.gz` prend toutes les années d'un coup, et `union_by_name = true` aligne les colonnes par leur nom, même si leur ordre change d'un fichier à l'autre.

In [ ]:
import duckdb
import matplotlib.pyplot as plt

con = duckdb.connect("../data/dvf.duckdb")   # base stockée dans un fichier (créé s'il n'existe pas)

con.sql("""
    CREATE OR REPLACE TABLE raw_dvf AS
    SELECT * FROM read_csv_auto('../data/raw/974_*.csv.gz', union_by_name = true)
""")

con.sql("SELECT COUNT(*) AS nb_lignes FROM raw_dvf").show()

In [ ]:
# Aperçu des premières lignes (.df() donne un tableau pandas, plus lisible que .show() avec 40 colonnes)
con.sql("SELECT * FROM raw_dvf LIMIT 5").df()

## Nombre de lignes et de mutations distinctes par année

In [ ]:
con.sql("""
    SELECT
        YEAR(date_mutation)          AS annee,
        COUNT(*)                     AS nb_lignes,
        COUNT(DISTINCT id_mutation)  AS nb_mutations
    FROM raw_dvf
    GROUP BY annee
    ORDER BY annee
""").df()

Il y a 2 à 3 fois plus de lignes que de mutations : c'est le piège DVF vu dans le notebook 1 (une vente = plusieurs lignes). Le nombre de mutations reste stable autour de 7 000 à 10 000 par an, avec un creux en 2024-2025.

## Réécrire `ventes_simples` en SQL, avec des CTE

Une CTE (`WITH nom AS (...)`) est une sous-requête nommée : on découpe le calcul en étapes lisibles, comme les variables de la fonction pandas.

1. `ventes` : uniquement les mutations de type `Vente` ;
2. `mutations_simples` : les ventes qui contiennent un seul local principal ;
3. `locaux` : ces locaux, avec une surface et un prix valides, et le calcul du prix au m² et du drapeau `hors_marche`.

In [ ]:
con.sql("""
    CREATE OR REPLACE TABLE ventes_simples AS
    WITH ventes AS (
        SELECT *
        FROM raw_dvf
        WHERE nature_mutation = 'Vente'
    ),
    mutations_simples AS (
        SELECT id_mutation
        FROM ventes
        WHERE type_local IN ('Maison', 'Appartement', 'Local industriel. commercial ou assimilé')
        GROUP BY id_mutation
        HAVING COUNT(*) = 1
    ),
    locaux AS (
        SELECT
            v.*,
            v.valeur_fonciere / v.surface_reelle_bati         AS prix_m2,
            v.valeur_fonciere / v.surface_reelle_bati < 500   AS hors_marche
        FROM ventes v
        INNER JOIN mutations_simples m ON v.id_mutation = m.id_mutation
        WHERE v.type_local IN ('Maison', 'Appartement', 'Local industriel. commercial ou assimilé')
          AND v.surface_reelle_bati > 0
          AND v.valeur_fonciere > 1000          -- exclut les ventes symboliques à 1 €
    )
    SELECT * FROM locaux
""")

con.sql("SELECT COUNT(*) AS nb_ventes, COUNT_IF(hors_marche) AS nb_hors_marche FROM ventes_simples").show()

## Vérification 1 : les effectifs

On compare avec le notebook 1, qui donnait pour 2024 **2 003 appartements et 2 171 maisons**.

In [ ]:
con.sql("""
    SELECT YEAR(date_mutation) AS annee, type_local, COUNT(*) AS nb
    FROM ventes_simples
    GROUP BY annee, type_local
    ORDER BY annee, type_local
""").df()

Pour 2024, on retrouve exactement 2 003 appartements et 2 171 maisons : la version SQL filtre comme la version pandas.

## Vérification 2 : les médianes

Même contrôle sur les prix : appartements en 2024, sans les ventes hors marché.

In [ ]:
con.sql("""
    SELECT nom_commune, MEDIAN(prix_m2) AS prix_median, COUNT(*) AS nb
    FROM ventes_simples
    WHERE NOT hors_marche
      AND YEAR(date_mutation) = 2024
      AND type_local = 'Appartement'
    GROUP BY nom_commune
    ORDER BY prix_median DESC
""").df()

Les médianes sont identiques à celles du notebook 1 (Saint-Paul ≈ 5 109 €/m², Saint-Leu ≈ 4 081 €/m², Saint-Pierre ≈ 3 264 €/m²). Les deux implémentations sont cohérentes.

## Évolution du prix médian au m² par année et par type

Pour chaque année et chaque type (Appartement, Maison) : prix médian au m² et nombre de ventes, sans les ventes hors marché.

In [ ]:
evolution = con.sql("""
    SELECT YEAR(date_mutation) AS annee, type_local, MEDIAN(prix_m2) AS prix_median, COUNT(*) AS nb
    FROM ventes_simples
    WHERE NOT hors_marche
      AND type_local IN ('Appartement', 'Maison')
    GROUP BY annee, type_local
    ORDER BY annee, type_local
""").df()
evolution

In [ ]:
pivot = evolution.pivot(index='annee', columns='type_local', values='prix_median')

ax = pivot.plot(marker='o', figsize=(8, 5))
ax.set_title("Prix médian au m² à La Réunion (ventes simples, hors marché exclues)")
ax.set_xlabel("Année")
ax.set_ylabel("€/m²")
ax.set_xticks(pivot.index)          # années entières uniquement
ax.grid(alpha=0.3)
ax.legend(title="")
plt.show()

- **Appartements** : hausse régulière, d'environ 2 300 à 2 860 €/m², soit +24 % en 4 ans.
- **Maisons** : forte hausse jusqu'en 2023, puis plateau autour de 2 830 €/m². Au total, environ +12 %.
- **En 2025, les courbes se croisent** : le m² d'appartement dépasse celui des maisons.

Rappel : pour les maisons, le prix au m² inclut la valeur du terrain (limite « bâti sans terrain » vue dans le notebook 1).

## Fermer la connexion

DuckDB ne laisse qu'un seul programme ouvrir le fichier `dvf.duckdb` en écriture. Tant que la connexion reste ouverte, aucun autre script ou notebook ne peut s'y connecter : on la ferme à la fin.

In [ ]:
con.close()